# Generate branches

Registry-driven generation of `<TECH>_<METRIC>_<TYPE>_<VERSION>` branches.

**Default flow:** set only `LANGUAGE` and `VERSION`, then run all cells → **412 branches**
(14 techniques × 103 metrics × 4 types) for Python.

### Expected runtime (approximate)

| Scope | Branches | Time |
|-------|----------|------|
| Full matrix (default) | 412 | 15–45 min (disk-bound) |
| Single technique (e.g. SA) | 24 | ~1–2 min |
| Single metric × 4 types | 4 | seconds |

Validation adds ~2–5 min for the full matrix. Use optional `TECHNIQUES` / `METRICS` / `TYPES`
overrides below for smaller partial runs.

`DO_GIT` / `DO_PUSH` remain opt-in (default `False`).

## Parameters

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config" / "metrics_registry.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib import metrics
from lib.registry import iter_branches, load_registry
from lib.generator import generate_branches, create_git_branches, push_branches
from lib.validate import validate_build_report

# --- Required inputs ---
LANGUAGE = "python"
VERSION = "2.6"

# --- Optional overrides (default = full 412-branch matrix) ---
TECHNIQUES = metrics.ALL_TECHNIQUES   # all 14 technique groups
METRICS = metrics.ALL_METRICS         # all metrics per technique
TYPES = list(metrics.FIXED_BRANCH_TYPES)  # Bug, BugFX, TCC, CC

BUILD_DIR = "build"
DO_GIT = False
DO_PUSH = False

registry = load_registry()

## Plan / preview (informational — does not block execution)

In [ ]:
planned = list(iter_branches(TECHNIQUES, METRICS, TYPES, VERSION, registry))
plan_count = len(planned)

by_tech = defaultdict(lambda: {"metrics": set(), "branches": 0})
for tech, metric, bt, name in planned:
    by_tech[tech]["metrics"].add(metric)
    by_tech[tech]["branches"] += 1

print(f"Plan: {plan_count} branches (language={LANGUAGE}, version={VERSION})")
print(f"  techniques override: {TECHNIQUES!r}  metrics override: {METRICS!r}  types: {TYPES}")
print()
print(f"{'Technique':<6} {'Metrics':>7} {'Branches':>9}")
print("-" * 26)
for tech in sorted(by_tech.keys()):
    row = by_tech[tech]
    print(f"{tech:<6} {len(row['metrics']):>7} {row['branches']:>9}")
print("-" * 26)
print(f"{'TOTAL':<6} {sum(len(r['metrics']) for r in by_tech.values()):>7} {plan_count:>9}")
print()
if plan_count <= 48:
    for tech, metric, bt, name in planned:
        print(f"  {name}")
else:
    for tech, metric, bt, name in planned[:8]:
        print(f"  {name}")
    print(f"  ... ({plan_count - 16} more) ...")
    for tech, metric, bt, name in planned[-8:]:
        print(f"  {name}")

## Generate

In [ ]:
generated, gen_errors = generate_branches(
    TECHNIQUES, METRICS, TYPES, VERSION, LANGUAGE, BUILD_DIR, str(ROOT),
    continue_on_error=True,
)
print(f"Generated {len(generated)} / {plan_count} branches under {ROOT / BUILD_DIR}")
if gen_errors:
    print(f"\nGeneration errors: {len(gen_errors)}")
    for err in gen_errors[:20]:
        print(f"  {err['branch']}: {err['error']}")
    if len(gen_errors) > 20:
        print(f"  ... and {len(gen_errors) - 20} more")

## Validate

In [ ]:
results = validate_build_report(
    TECHNIQUES, METRICS, TYPES, VERSION, LANGUAGE, BUILD_DIR, str(ROOT))

pass_n = sum(1 for r in results if r[5] == "PASS")
fail_n = sum(1 for r in results if r[5] == "FAIL")
miss_n = sum(1 for r in results if r[5] == "MISSING")

print(f"{'Branch':<32} {'Tech':<4} {'Met':<5} {'Type':<6} {'LOC':>6}  Status")
print("-" * 72)
for name, tech, metric, bt, loc, status, path, msg in results:
    print(f"{name:<32} {tech:<4} {metric:<5} {bt:<6} {loc:>6}  {status}")

print(f"\nValidation: {pass_n} PASS, {fail_n} FAIL, {miss_n} MISSING (of {len(results)} planned)")
if fail_n or miss_n:
    print("\nFailures:")
    for name, tech, metric, bt, loc, status, path, msg in results:
        if status != "PASS":
            print(f"  {name} [{status}]: {msg}")

## Optional git materialize

In [ ]:
if DO_GIT:
    if DO_PUSH:
        print("WARNING: DO_PUSH will force-push branches to origin")
    create_git_branches(TECHNIQUES, METRICS, TYPES, VERSION, LANGUAGE, str(ROOT), BUILD_DIR)
    if DO_PUSH:
        push_branches(generated, str(ROOT))
else:
    print("Skipped git (DO_GIT=False)")

## Summary + build_manifest.json

In [ ]:
manifest = {
    "language": LANGUAGE,
    "version": VERSION,
    "techniques": TECHNIQUES,
    "metrics": METRICS,
    "types": TYPES,
    "planned_count": plan_count,
    "generated_count": len(generated),
    "generation_errors": gen_errors,
    "branches": generated,
    "validation": [
        {
            "branch": r[0], "technique": r[1], "metric": r[2], "type": r[3],
            "loc": r[4], "status": r[5], "path": r[6], "message": r[7],
        }
        for r in results
    ],
}
manifest_path = ROOT / "build_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote {manifest_path} ({len(manifest['validation'])} validation entries)")